# Sliding Window Cohort V2

## 주요 개선사항
1. ✅ DNR 환자 포함 (window-level 필터링)
2. ✅ 조기 이벤트: 환자 제외 → window 제외로 변경
3. ✅ Stride: 6h → 2h (샘플 수 대폭 증가)
4. ✅ Event censoring: event 이전 windows만 사용

## 예상 결과
- 총 샘플: 194K → 500K-900K
- 고유 환자: 36K → 54K (DNR 포함)
- Event prevalence: 2.85% → 8-12%

In [1]:
import duckdb
import pandas as pd
import os
from datetime import datetime

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [2]:
# DuckDB 연결
db_path = '../data/duckdb/mimic_total.duckdb'
con = duckdb.connect(db_path)

print("=== Sliding Window Cohort V2 생성 시작 ===\n")

=== Sliding Window Cohort V2 생성 시작 ===



## Step 1: 기본 코호트 (환자 제외 없음)

In [3]:
print("Step 1: 기본 포함 기준 적용 (성인, 첫 입실, 24h+ 체류)")

base_cohort_query = """
CREATE OR REPLACE TABLE cohort_base_v2 AS
SELECT 
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    CAST(i.intime AS TIMESTAMP) as intime,
    CAST(i.outtime AS TIMESTAMP) as outtime,
    CAST(i.los AS DOUBLE) as los,
    i.first_careunit,
    i.last_careunit,
    CAST(p.anchor_age AS INTEGER) as anchor_age,
    p.gender,
    CAST(p.dod AS TIMESTAMP) as dod,
    CAST(a.admittime AS TIMESTAMP) as admittime,
    CAST(a.dischtime AS TIMESTAMP) as dischtime,
    CAST(a.deathtime AS TIMESTAMP) as deathtime,
    a.hospital_expire_flag,
    ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) as icu_seq
FROM icustays i
INNER JOIN patients p ON i.subject_id = p.subject_id
INNER JOIN admissions a ON i.hadm_id = a.hadm_id
WHERE 
    CAST(p.anchor_age AS INTEGER) >= 18
    AND CAST(i.los AS DOUBLE) >= 1.0
"""

con.execute(base_cohort_query)

# 첫 번째 입실만
con.execute("""
CREATE OR REPLACE TABLE cohort_first_stay_v2 AS
SELECT * FROM cohort_base_v2
WHERE icu_seq = 1
""")

result = con.execute("SELECT COUNT(*) as count FROM cohort_first_stay_v2").df()
print(f"기본 포함 기준 후: {result['count'][0]:,}명\n")

Step 1: 기본 포함 기준 적용 (성인, 첫 입실, 24h+ 체류)
기본 포함 기준 후: 54,551명



## Step 2: DNR 시간 저장 (환자 제외 안 함)

In [4]:
print("Step 2: DNR 시간 저장 (window-level 필터링을 위해)")

dnr_query = """
CREATE OR REPLACE TABLE dnr_times AS
SELECT 
    c.stay_id,
    MIN(CAST(ce.charttime AS TIMESTAMP)) as dnr_time
FROM cohort_first_stay_v2 c
INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
WHERE 
    ce.itemid = '223758'
GROUP BY c.stay_id
"""

con.execute(dnr_query)

dnr_count = con.execute("SELECT COUNT(*) as count FROM dnr_times").fetchone()[0]
print(f"DNR 발생 환자: {dnr_count:,}명")
print("  → 이전: 전체 제외")
print("  → 이후: DNR 이전 windows만 사용\n")

Step 2: DNR 시간 저장 (window-level 필터링을 위해)
DNR 발생 환자: 25,629명
  → 이전: 전체 제외
  → 이후: DNR 이전 windows만 사용



## Step 3: Event 시간 저장

In [5]:
print("Step 3: Event 시간 저장 (ventilation, pressor)")

# Ventilation start times
vent_query = """
CREATE OR REPLACE TABLE vent_start_times_v2 AS
SELECT 
    stay_id,
    MIN(CAST(starttime AS TIMESTAMP)) as vent_start
FROM procedureevents
WHERE 
    itemid = '225792'
    AND starttime IS NOT NULL
GROUP BY stay_id
"""

con.execute(vent_query)

# Pressor start times
pressor_query = """
CREATE OR REPLACE TABLE pressor_start_times_v2 AS
SELECT 
    stay_id,
    MIN(CAST(starttime AS TIMESTAMP)) as pressor_start
FROM inputevents
WHERE 
    itemid IN ('221906', '221289', '222315', '221662')
    AND starttime IS NOT NULL
    AND rate IS NOT NULL
    AND CAST(rate AS DOUBLE) > 0
GROUP BY stay_id
"""

con.execute(pressor_query)

vent_count = con.execute("SELECT COUNT(*) as count FROM vent_start_times_v2").fetchone()[0]
pressor_count = con.execute("SELECT COUNT(*) as count FROM pressor_start_times_v2").fetchone()[0]

print(f"Ventilation 시작 기록: {vent_count:,}명")
print(f"Pressor 시작 기록: {pressor_count:,}명\n")

Step 3: Event 시간 저장 (ventilation, pressor)
Ventilation 시작 기록: 31,969명
Pressor 시작 기록: 18,500명



## Step 4: Sliding Windows 생성 (1h stride)

In [6]:
print("Step 4: Sliding Windows 생성")
print("  - Window size: 6h")
print("  - Stride: 1h")
print("  - Start: ICU + 6h")
print("  - Max: 72h\n")

sliding_window_query = """
CREATE OR REPLACE TABLE cohort_sliding_window_v2 AS
WITH base_cohort AS (
    SELECT 
        c.*,
        d.dnr_time,
        v.vent_start,
        p.pressor_start,
        
        -- ICU mortality
        CASE 
            WHEN c.deathtime IS NOT NULL 
            AND c.deathtime <= c.outtime
            THEN 1 ELSE 0
        END as icu_mortality,
        
        -- Hospital mortality
        CASE 
            WHEN c.hospital_expire_flag = '1' THEN 1
            ELSE 0
        END as hospital_mortality
        
    FROM cohort_first_stay_v2 c
    LEFT JOIN dnr_times d ON c.stay_id = d.stay_id
    LEFT JOIN vent_start_times_v2 v ON c.stay_id = v.stay_id
    LEFT JOIN pressor_start_times_v2 p ON c.stay_id = p.stay_id
),

-- 2시간 간격으로 windows 생성 (6h부터 72h까지)
time_windows AS (
    SELECT UNNEST(GENERATE_SERIES(6, 72, 1)) as observation_hour
),

-- 각 환자 × 시간 조합 생성
candidate_windows AS (
    SELECT 
        bc.*,
        tw.observation_hour,
        bc.intime + (tw.observation_hour || ' hours')::INTERVAL as observation_end_time,
        bc.intime + ((tw.observation_hour - 6) || ' hours')::INTERVAL as observation_start_time
    FROM base_cohort bc
    CROSS JOIN time_windows tw
),

-- Window-level 필터링
valid_windows AS (
    SELECT cw.*
    FROM candidate_windows cw
    WHERE
        -- 1. ICU에 있어야 함
        cw.observation_end_time <= cw.outtime
        
        -- 2. DNR 필터: DNR 이전 windows만
        AND (
            cw.dnr_time IS NULL 
            OR cw.observation_end_time < cw.dnr_time
        )
        
        -- 3. Event censoring: event 이전 windows만
        AND (
            cw.deathtime IS NULL 
            OR cw.observation_end_time < cw.deathtime
        )
        AND (
            cw.vent_start IS NULL 
            OR cw.observation_end_time < cw.vent_start
        )
        AND (
            cw.pressor_start IS NULL 
            OR cw.observation_end_time < cw.pressor_start
        )
        
        -- 4. 필수 활력징후 존재 확인 (window 내)
        AND EXISTS (
            SELECT 1
            FROM chartevents ce
            WHERE ce.stay_id = cw.stay_id
            AND ce.itemid IN (
                '220045', '220210',  -- Heart Rate
                '220050', '220051', '220052',  -- ABP
                '220179', '220180', '220181'   -- NIBP
            )
            AND CAST(ce.charttime AS TIMESTAMP) >= cw.observation_start_time
            AND CAST(ce.charttime AS TIMESTAMP) <= cw.observation_end_time
        )
)

-- Labels 계산
SELECT 
    vw.*,
    
    -- === 6시간 예측 ===
    CASE 
        WHEN vw.deathtime IS NOT NULL
        AND vw.deathtime > vw.observation_end_time
        AND vw.deathtime <= vw.observation_end_time + INTERVAL '6 hours'
        THEN 1 ELSE 0
    END as death_next_6h,
    
    CASE 
        WHEN vw.vent_start IS NOT NULL
        AND vw.vent_start > vw.observation_end_time
        AND vw.vent_start <= vw.observation_end_time + INTERVAL '6 hours'
        THEN 1 ELSE 0
    END as vent_start_next_6h,
    
    CASE 
        WHEN vw.pressor_start IS NOT NULL
        AND vw.pressor_start > vw.observation_end_time
        AND vw.pressor_start <= vw.observation_end_time + INTERVAL '6 hours'
        THEN 1 ELSE 0
    END as pressor_start_next_6h,
    
    -- === 12시간 예측 ===
    CASE 
        WHEN vw.deathtime IS NOT NULL
        AND vw.deathtime > vw.observation_end_time
        AND vw.deathtime <= vw.observation_end_time + INTERVAL '12 hours'
        THEN 1 ELSE 0
    END as death_next_12h,
    
    CASE 
        WHEN vw.vent_start IS NOT NULL
        AND vw.vent_start > vw.observation_end_time
        AND vw.vent_start <= vw.observation_end_time + INTERVAL '12 hours'
        THEN 1 ELSE 0
    END as vent_start_next_12h,
    
    CASE 
        WHEN vw.pressor_start IS NOT NULL
        AND vw.pressor_start > vw.observation_end_time
        AND vw.pressor_start <= vw.observation_end_time + INTERVAL '12 hours'
        THEN 1 ELSE 0
    END as pressor_start_next_12h,
    
    -- === 24시간 예측 ===
    CASE 
        WHEN vw.deathtime IS NOT NULL
        AND vw.deathtime > vw.observation_end_time
        AND vw.deathtime <= vw.observation_end_time + INTERVAL '24 hours'
        THEN 1 ELSE 0
    END as death_next_24h,
    
    CASE 
        WHEN vw.vent_start IS NOT NULL
        AND vw.vent_start > vw.observation_end_time
        AND vw.vent_start <= vw.observation_end_time + INTERVAL '24 hours'
        THEN 1 ELSE 0
    END as vent_start_next_24h,
    
    CASE 
        WHEN vw.pressor_start IS NOT NULL
        AND vw.pressor_start > vw.observation_end_time
        AND vw.pressor_start <= vw.observation_end_time + INTERVAL '24 hours'
        THEN 1 ELSE 0
    END as pressor_start_next_24h,
    
    -- === Composite labels ===
    CASE 
        WHEN (
            (vw.deathtime IS NOT NULL
             AND vw.deathtime > vw.observation_end_time
             AND vw.deathtime <= vw.observation_end_time + INTERVAL '6 hours')
            OR
            (vw.vent_start IS NOT NULL
             AND vw.vent_start > vw.observation_end_time
             AND vw.vent_start <= vw.observation_end_time + INTERVAL '6 hours')
            OR
            (vw.pressor_start IS NOT NULL
             AND vw.pressor_start > vw.observation_end_time
             AND vw.pressor_start <= vw.observation_end_time + INTERVAL '6 hours')
        ) THEN 1 ELSE 0
    END as composite_next_6h,
    
    CASE 
        WHEN (
            (vw.deathtime IS NOT NULL
             AND vw.deathtime > vw.observation_end_time
             AND vw.deathtime <= vw.observation_end_time + INTERVAL '12 hours')
            OR
            (vw.vent_start IS NOT NULL
             AND vw.vent_start > vw.observation_end_time
             AND vw.vent_start <= vw.observation_end_time + INTERVAL '12 hours')
            OR
            (vw.pressor_start IS NOT NULL
             AND vw.pressor_start > vw.observation_end_time
             AND vw.pressor_start <= vw.observation_end_time + INTERVAL '12 hours')
        ) THEN 1 ELSE 0
    END as composite_next_12h,
    
    CASE 
        WHEN (
            (vw.deathtime IS NOT NULL
             AND vw.deathtime > vw.observation_end_time
             AND vw.deathtime <= vw.observation_end_time + INTERVAL '24 hours')
            OR
            (vw.vent_start IS NOT NULL
             AND vw.vent_start > vw.observation_end_time
             AND vw.vent_start <= vw.observation_end_time + INTERVAL '24 hours')
            OR
            (vw.pressor_start IS NOT NULL
             AND vw.pressor_start > vw.observation_end_time
             AND vw.pressor_start <= vw.observation_end_time + INTERVAL '24 hours')
        ) THEN 1 ELSE 0
    END as composite_next_24h

FROM valid_windows vw
"""

con.execute(sliding_window_query)

print("✓ Sliding windows 생성 완료\n")

Step 4: Sliding Windows 생성
  - Window size: 6h
  - Stride: 1h
  - Start: ICU + 6h
  - Max: 72h

✓ Sliding windows 생성 완료



## Step 5: 통계 확인

In [7]:
print("=== V2 결과 통계 ===\n")

# 전체 통계
total_stats = con.execute("""
    SELECT 
        COUNT(*) as total_samples,
        COUNT(DISTINCT stay_id) as unique_patients,
        ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT stay_id), 1) as avg_windows_per_patient
    FROM cohort_sliding_window_v2
""").df()

print("전체 통계:")
print(f"  총 샘플: {total_stats['total_samples'][0]:,}개")
print(f"  고유 환자: {total_stats['unique_patients'][0]:,}명")
print(f"  환자당 평균 windows: {total_stats['avg_windows_per_patient'][0]}개\n")

# 시간대별 통계
stats = con.execute("""
    SELECT 
        observation_hour,
        COUNT(*) as n_samples,
        SUM(death_next_6h) as deaths_6h,
        SUM(death_next_12h) as deaths_12h,
        SUM(death_next_24h) as deaths_24h,
        SUM(vent_start_next_6h) as vent_6h,
        SUM(pressor_start_next_6h) as pressor_6h,
        SUM(composite_next_6h) as composite_6h,
        SUM(composite_next_12h) as composite_12h,
        SUM(composite_next_24h) as composite_24h,
        COUNT(DISTINCT stay_id) as unique_patients
    FROM cohort_sliding_window_v2
    GROUP BY observation_hour
    ORDER BY observation_hour
""").df()

print("시간대별 통계:")
print(stats.head(10))
print("\n")

# Event prevalence
print("=== Event Prevalence (첫 3개 시점) ===\n")
for _, row in stats.head(3).iterrows():
    hour = int(row['observation_hour'])
    n = row['n_samples']
    print(f"[{hour}시간 시점]")
    print(f"  총 샘플: {n:,}개")
    print(f"  향후 6h 사망: {row['deaths_6h']} ({row['deaths_6h']/n*100:.2f}%)")
    print(f"  향후 6h 통합: {row['composite_6h']} ({row['composite_6h']/n*100:.2f}%)")
    print(f"  향후 12h 통합: {row['composite_12h']} ({row['composite_12h']/n*100:.2f}%)")
    print(f"  향후 24h 통합: {row['composite_24h']} ({row['composite_24h']/n*100:.2f}%)\n")

=== V2 결과 통계 ===

전체 통계:
  총 샘플: 934,312개
  고유 환자: 23,390명
  환자당 평균 windows: 39.9개

시간대별 통계:
   observation_hour  n_samples  deaths_6h  deaths_12h  deaths_24h  vent_6h  \
0                 6      23220        2.0         6.0       102.0    423.0   
1                 7      22943        1.0         6.0       108.0    391.0   
2                 8      22648        1.0         5.0       116.0    347.0   
3                 9      22406        3.0         8.0       122.0    325.0   
4                10      22194        4.0        12.0       130.0    322.0   
5                11      21992        4.0        18.0       136.0    318.0   
6                12      21817        4.0        23.0       142.0    308.0   
7                13      21634        4.0        28.0       146.0    290.0   
8                14      21479        3.0        39.0       159.0    297.0   
9                15      21302        4.0        48.0       162.0    273.0   

   pressor_6h  composite_6h  composite_12h  comp

## Step 6: V1 vs V2 비교

In [8]:
print("=== V1 vs V2 비교 ===\n")

# V1이 있는 경우에만 비교
v1_exists = con.execute("""
    SELECT COUNT(*) as count 
    FROM information_schema.tables 
    WHERE table_name = 'cohort_sliding_window'
""").fetchone()[0]

if v1_exists > 0:
    comparison = con.execute("""
        SELECT 
            'V1' as version,
            COUNT(*) as total_samples,
            COUNT(DISTINCT stay_id) as unique_patients,
            ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT stay_id), 1) as avg_windows,
            SUM(composite_next_6h) as composite_6h,
            ROUND(CAST(SUM(composite_next_6h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_6h_pct
        FROM cohort_sliding_window
        
        UNION ALL
        
        SELECT 
            'V2' as version,
            COUNT(*) as total_samples,
            COUNT(DISTINCT stay_id) as unique_patients,
            ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT stay_id), 1) as avg_windows,
            SUM(composite_next_6h) as composite_6h,
            ROUND(CAST(SUM(composite_next_6h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_6h_pct
        FROM cohort_sliding_window_v2
    """).df()
    
    print(comparison)
    print("\n")
    
    # Improvement 계산
    v1_samples = comparison.loc[0, 'total_samples']
    v2_samples = comparison.loc[1, 'total_samples']
    improvement = (v2_samples - v1_samples) / v1_samples * 100
    
    print(f"샘플 수 증가: {v1_samples:,} → {v2_samples:,} (+{improvement:.1f}%)")
else:
    print("V1 테이블 없음 (비교 생략)\n")

=== V1 vs V2 비교 ===

  version  total_samples  unique_patients  avg_windows  composite_6h  \
0      V1         194641            36400          5.3        3477.0   
1      V2         934312            23390         39.9       11931.0   

   composite_6h_pct  
0              1.79  
1              1.28  


샘플 수 증가: 194,641 → 934,312 (+380.0%)


## Step 7: DNR 환자 포함 확인

In [9]:
print("=== DNR 환자 포함 확인 ===\n")

dnr_in_cohort = con.execute("""
    SELECT COUNT(DISTINCT c.stay_id) as dnr_patients_included
    FROM cohort_sliding_window_v2 c
    INNER JOIN dnr_times d ON c.stay_id = d.stay_id
""").fetchone()[0]

total_dnr = con.execute("SELECT COUNT(*) as count FROM dnr_times").fetchone()[0]

print(f"전체 DNR 환자: {total_dnr:,}명")
print(f"V2 코호트에 포함된 DNR 환자: {dnr_in_cohort:,}명")
print(f"포함률: {dnr_in_cohort/total_dnr*100:.1f}%\n")

# DNR 환자 샘플 확인
dnr_sample = con.execute("""
    SELECT 
        c.stay_id,
        c.observation_hour,
        c.observation_end_time,
        d.dnr_time,
        EXTRACT(EPOCH FROM (d.dnr_time - c.observation_end_time)) / 3600 as hours_before_dnr
    FROM cohort_sliding_window_v2 c
    INNER JOIN dnr_times d ON c.stay_id = d.stay_id
    WHERE c.stay_id = (
        SELECT stay_id FROM dnr_times LIMIT 1
    )
    ORDER BY c.observation_hour
    LIMIT 5
""").df()

print("DNR 환자 샘플 (windows는 DNR 이전만):")
print(dnr_sample)

=== DNR 환자 포함 확인 ===

전체 DNR 환자: 25,629명
V2 코호트에 포함된 DNR 환자: 4,267명
포함률: 16.6%

DNR 환자 샘플 (windows는 DNR 이전만):
Empty DataFrame
Columns: [stay_id, observation_hour, observation_end_time, dnr_time, hours_before_dnr]
Index: []


## Step 8: CSV 저장

In [10]:
print("\n" + "="*60)
print("CSV 파일 저장")
print("="*60 + "\n")

output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# 1. 메인 코호트 테이블
print("1. 메인 코호트 테이블 저장 중...")
output_path = os.path.join(output_dir, 'cohort_sliding_window_v2.csv')

con.execute(f"""
    COPY cohort_sliding_window_v2 
    TO '{output_path}' 
    (HEADER, DELIMITER ',')
""")

file_size = os.path.getsize(output_path) / (1024 * 1024)
row_count = con.execute("SELECT COUNT(*) as count FROM cohort_sliding_window_v2").fetchone()[0]
unique_patients = con.execute("SELECT COUNT(DISTINCT stay_id) as count FROM cohort_sliding_window_v2").fetchone()[0]
col_count = len(con.execute("DESCRIBE cohort_sliding_window_v2").df())

print(f"✓ 저장 완료: cohort_sliding_window_v2.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 총 행 수: {row_count:,}개")
print(f"  - 고유 환자: {unique_patients:,}명")
print(f"  - 환자당 평균 시점: {row_count/unique_patients:.1f}개")
print(f"  - 컬럼 수: {col_count}개\n")

# 2. 통계 요약
print("2. 통계 요약 저장 중...")
summary_path = os.path.join(output_dir, 'cohort_summary_statistics_v2.csv')

summary_df = con.execute("""
    SELECT 
        observation_hour,
        COUNT(*) as n_samples,
        COUNT(DISTINCT stay_id) as unique_patients,
        SUM(death_next_6h) as deaths_6h,
        SUM(death_next_12h) as deaths_12h,
        SUM(death_next_24h) as deaths_24h,
        SUM(vent_start_next_6h) as vent_6h,
        SUM(vent_start_next_12h) as vent_12h,
        SUM(vent_start_next_24h) as vent_24h,
        SUM(pressor_start_next_6h) as pressor_6h,
        SUM(pressor_start_next_12h) as pressor_12h,
        SUM(pressor_start_next_24h) as pressor_24h,
        SUM(composite_next_6h) as composite_6h,
        SUM(composite_next_12h) as composite_12h,
        SUM(composite_next_24h) as composite_24h,
        ROUND(CAST(SUM(death_next_6h) AS DOUBLE) / COUNT(*) * 100, 2) as death_6h_pct,
        ROUND(CAST(SUM(composite_next_6h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_6h_pct,
        ROUND(CAST(SUM(composite_next_12h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_12h_pct,
        ROUND(CAST(SUM(composite_next_24h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_24h_pct
    FROM cohort_sliding_window_v2
    GROUP BY observation_hour
    ORDER BY observation_hour
""").df()

summary_df.to_csv(summary_path, index=False)
print(f"✓ 저장 완료: cohort_summary_statistics_v2.csv\n")

# 3. 컬럼 정보
print("3. 컬럼 정보 저장 중...")
columns_path = os.path.join(output_dir, 'cohort_columns_info_v2.csv')

columns_df = con.execute("DESCRIBE cohort_sliding_window_v2").df()
columns_df.to_csv(columns_path, index=False)
print(f"✓ 저장 완료: cohort_columns_info_v2.csv\n")

print("=== Sliding Window Cohort V2 생성 완료 ===")
print(f"테이블명: cohort_sliding_window_v2")
print(f"파일 경로: {output_dir}")


CSV 파일 저장

1. 메인 코호트 테이블 저장 중...
✓ 저장 완료: cohort_sliding_window_v2.csv
  - 파일 크기: 255.83 MB
  - 총 행 수: 934,312개
  - 고유 환자: 23,390명
  - 환자당 평균 시점: 39.9개
  - 컬럼 수: 36개

2. 통계 요약 저장 중...
✓ 저장 완료: cohort_summary_statistics_v2.csv

3. 컬럼 정보 저장 중...
✓ 저장 완료: cohort_columns_info_v2.csv

=== Sliding Window Cohort V2 생성 완료 ===
테이블명: cohort_sliding_window_v2
파일 경로: ../data/processed
